# Modelování horního chvostu rozměrové odchylky pomocí PROC QUANTREG

## Shrnutí

Provoz přesného obrábění se zajímá o **nejhorší případ** rozměrové odchylky mezi díly, ne jen o průměr, protože horní chvost rozdělení řídí zmetkovitost a reklamace zákazníků. Tento sešit používá **PROC QUANTREG** k proložení kvantilové regrese pro medián a 90. percentil a ukazuje, že opotřebení nástroje, rychlost cyklu a posuv působí na horní kvantil (riziko zmetků) mnohem silněji než na medián — typický znak heteroskedastického procesu, kde s opotřebením nástroje roste variabilita.

## Zdroje dat

| Datová sada | Řádky | Popis | Klíčové proměnné |
|---------|------|-------------|---------------|
| `work.machining` | 100 | Syntetické záznamy na úrovni dílu ze soustružnické linky CNC, generované přímo pomocí `call streaminit`/`rand`. Rozměrová odchylka (mikrony) od jmenovité hodnoty je modelována pomocí heteroskedastické chyby, jejíž rozptyl roste s opotřebením nástroje a rychlostí cyklu, takže procesní faktory působí silněji na horní chvost než na medián. | `Deviation` (odezva, mikrony), `ToolWear` (kumulativní řezné minuty), `CycleSpeed` (ot/min), `CoolantTemp` (°C), `Humidity` (%RH), `FeedRate` (mm/ot), `Machine` (TŘÍDA: M1–M4), `Shift` (TŘÍDA: Day/Eve/Night), `PartID` |

# Modelování procesních faktorů horního chvostu rozměrové odchylky

## Případ použití ve výrobě: modelování rizika zmetků na soustružnické lince CNC

Na lince přesného obrábění jsou díly vyřazovány, když **rozměrová odchylka** od jmenovité hodnoty příliš vzroste. Podnik neztrácí peníze na *průměrném* dílu — ztrácí je na **horním chvostu** rozdělení odchylky. Klasická lineární regrese modeluje podmíněný průměr a může zcela přehlédnout faktory, které se projeví až ve chvíli, kdy se něco pokazí.

**Kvantilová regrese** modeluje zvolený podmíněný kvantil (například 90. percentil odchylky) místo průměru. **PROC QUANTREG** proloží jeden nebo více kvantilů v jediném volání a pro každý faktor a každý kvantil uvádí odhad parametru, jeho směrodatnou chybu a interval spolehlivosti. Postupujeme takto:

1. Vygenerujeme realistickou syntetickou datovou sadu na úrovni dílu, jejíž rozptyl chyby roste s opotřebením nástroje a rychlostí cyklu (aby faktory působily na chvost silněji než na střed).
2. Proložíme **medián** (q = 0,50) a **chvost rizika zmetků** (q = 0,90) jediným voláním PROC QUANTREG.
3. Porovnáme obě sady koeficientů vedle sebe, abychom ukázali, jak se sklony mění od středu ke chvostu.
4. Ohodnotíme každý díl jeho predikcí 90. percentilu, aby analytici mohli označit díly s vysokým rizikem chvostu.

Vše níže je samostatné — žádné externí soubory, žádná síť.

## Krok 1 — Vygenerování syntetických dat obrábění

Simulujeme soustružené díly napříč 4 stroji a 3 směnami. Klíčovým trikem reálnosti je **heteroskedasticita**: směrodatná odchylka náhodné chyby (`sigma`) roste s `ToolWear` a `CycleSpeed`. Díky tomu tyto dva faktory působí mnohem silněji na 90. percentil `Deviation` než na její medián — přesně tato situace je důvodem, proč se kvantilová regrese vyplatí. `Humidity` nenese v generujícím procesu žádný skutečný signál, takže slouží jako placebo faktor, který můžeme sledovat.

In [1]:
data work.machining;
    CALL streaminit(20260531);
    DÉLKA Machine $2 Shift $5;
    OPAKUJ PartID = 1 TO 100;
        /* Kategoriální faktory (TŘÍDA) */
        m = rand('integer', 1, 4);
        Machine = cats('M', put(m, 1.));
        s = rand('integer', 1, 3);
        KDYŽ s = 1 PAK Shift = 'Day';
        JINAK KDYŽ s = 2 PAK Shift = 'Eve';
        JINAK Shift = 'Night';

        /* Spojité procesní faktory */
        ToolWear     = rand('uniform') * 120;            /* kumulativní řezné minuty */
        CycleSpeed   = 1200 + rand('normal') * 180;      /* otáčky vřetena (ot/min) */
        CoolantTemp  = 22 + rand('normal') * 3;          /* °C */
        Humidity     = 45 + rand('normal') * 8;          /* %RH (placebo) */
        FeedRate     = 0.18 + rand('uniform') * 0.07;    /* mm/ot */

        /* Posuny podle stroje: jeden stroj běží tepleji */
        machoff = (Machine = 'M3') * 2.0 + (Machine = 'M4') * 0.8;
        /* Noční směna mírně posouvá hodnotu */
        shiftoff = (Shift = 'Night') * 1.2;

        /* Poloha (střední hodnota) odchylky v mikronech */
        MU = 5
           + 0.030 * ToolWear
           + 0.0015 * (CycleSpeed - 1200)
           + 0.45 * (CoolantTemp - 22)
           + 6.0 * FeedRate
           + machoff + shiftoff;

        /* Heteroskedastický rozptyl: chvost se rozšiřuje s opotřebením a rychlostí */
        SIGMA = 1.0
              + 0.020 * ToolWear
              + 0.0010 * abs(CycleSpeed - 1200);

        Deviation = MU + SIGMA * rand('normal');
        KDYŽ Deviation < 0 PAK Deviation = 0;

        PONECHAT PartID Machine Shift ToolWear CycleSpeed CoolantTemp
             Humidity FeedRate Deviation;
        VÝSTUP;
    KONEC;
SPUSTIT;


NOTE: DATA work.machining


NOTE: Wrote work.machining (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.03 seconds
  cpu   0.03 seconds


### Rychlý pohled na surové rozdělení

Před modelováním ověříme, že odezva je kladně zešikmená s významným horním chvostem — to je část rozdělení, která řídí zmetkovitost. Vytiskneme medián a horní percentily přímo z PROC UNIVARIATE, abychom viděli, o kolik je 90. percentil výše než medián.

In [2]:
PROCEDURA UNIVARIATE data=work.machining NOPRINT;
    PROMĚNNÁ Deviation;
    VÝSTUP out=work.devpct
        MEDIAN=Med p90=P90 p95=P95 p99=P99;
SPUSTIT;

PROCEDURA TISK data=work.devpct noobs ŠTÍTEK;
    ŠTÍTEK Med = 'Medián odchylky'
          P90 = '90. percentil'
          P95 = '95. percentil'
          P99 = '99. percentil';
SPUSTIT;


 Medián odchylky  90. percentil  95. percentil  99. percentil
----------------  -------------  -------------  -------------
    8.7251490709  12.4132063767  13.5691645665  17.0510365805




NOTE: PROC UNIVARIATE
NOTE: Output dataset work.devpct has 1 observations and 4 variables.
NOTE: PROC PRINT data=work.devpct

NOTE: PROC PRINT completed: 1 observations printed, 4 variables


## Krok 2 — Společné proložení mediánu a chvostu rizika zmetků

PROC QUANTREG proloží oba kvantily v jediném volání: `QUANTILE=0.5 0.90`. Příkaz `CLASS` deklaruje kategoriální procesní faktory (`Machine`, `Shift`); příkaz `MODEL` uvádí všechny kandidátní spojité efekty. Požadujeme `CI=SPARSITY`, což používá odhad založený na funkci řídkosti k výpočtu směrodatné chyby a 95% intervalu spolehlivosti pro každý koeficient.

Oba bloky výstupu čtěte jako před/po: první blok (q = 0,50) popisuje *typický* díl; druhý (q = 0,90) popisuje *díl náchylný ke zmetkovitosti*. Sledujte `ToolWear`, `CycleSpeed` a `FeedRate` — protože simulovaný rozptyl chyby roste s opotřebením a rychlostí, jejich sklony by měly být vyšší v horním kvantilu.

In [3]:
PROCEDURA quantreg data=work.machining ci=sparsity;
    TŘÍDA Machine Shift;
    MODEL Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
SPUSTIT;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Deviation

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -2.4433       2.0123      -6.3874       1.5007
MACHINE M3            1.3258       0.3574       0.6254       2.0262
MACHINE M2           -0.1735       0.3245      -0.8095       0.4625
MACHINE M1           -0.5599       0.3173      -1.1818       0.0619
SHIFT EVE            -1.6360       0.2964      -2.2170      -1.0550
SHIFT DAY            -0.9975       0.2909      -1.5676      -0.4275
TOOLWEAR              0.0240       0.0033       0.0174       0.0305
CYCLESPEED           -0.0007       0.0008      -0.0022       0.0009
COOLANTTEMP           0.4542       0.0395       0.3767       0.5316
HUMIDITY              0.0575       0.0150       0.0281       0.0868
FEEDRATE             -5.0538       5.8578     -16.5350       6.4273
Intercept           -12.8502       0.7036     -14.2292     -11.4711
MACHINE M3            1


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.


## Krok 3 — Porovnání středu a chvostu vedle sebe

Čtení dvou bloků koeficientů souběžně je nepohodlné, proto zachytíme tabulku parametrů pomocí `ODS OUTPUT ParameterEstimates=` (která obsahuje sloupec `Quantile`) a sloučíme odhady pro q = 0,50 a q = 0,90 pro spojité faktory do jedné srovnávací tabulky. Sloupec `Tail - Median` je hlavní číslo: velká kladná hodnota znamená, že faktor tlačí chvost rizika zmetků mnohem více, než posouvá typický díl.

In [4]:
ODS VÝSTUP ParameterEstimates=work.pe;
PROCEDURA quantreg data=work.machining ci=sparsity;
    TŘÍDA Machine Shift;
    MODEL Deviation = ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
SPUSTIT;

/* Sloučení sklonů pro medián a chvost u každého spojitého faktoru */
data work.COMPARE;
    PONECHAT Parameter MedianSlope TailSlope TailMinusMedian;
    SLOUČIT
        work.pe(KDE=(Quantile=0.5)
                PONECHAT=Quantile Parameter ESTIMATE
                PŘEJMENOVAT=(ESTIMATE=MedianSlope))
        work.pe(KDE=(Quantile=0.9)
                PONECHAT=Quantile Parameter ESTIMATE
                PŘEJMENOVAT=(ESTIMATE=TailSlope));
    PODLE Parameter;
    TailMinusMedian = TailSlope - MedianSlope;
SPUSTIT;

PROCEDURA TISK data=work.COMPARE noobs ŠTÍTEK;
    ŠTÍTEK Parameter       = 'Faktor'
          MedianSlope     = 'Sklon @ q=0,50'
          TailSlope       = 'Sklon @ q=0,90'
          TailMinusMedian = 'Chvost minus medián';
    FORMÁT MedianSlope TailSlope TailMinusMedian 10.4;
SPUSTIT;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Deviation

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -4.4131       2.7370      -9.7776       0.9514
TOOLWEAR              0.0146       0.0045       0.0057       0.0235
CYCLESPEED           -0.0000       0.0011      -0.0021       0.0021
COOLANTTEMP           0.4838       0.0528       0.3802       0.5873
HUMIDITY              0.0678       0.0203       0.0280       0.1076
FEEDRATE             -6.3520       7.7910     -21.6221       8.9181
Intercept            -5.3307       1.2671      -7.8142      -2.8471
TOOLWEAR              0.0416       0.0021       0.0375       0.0457
CYCLESPEED            0.0008       0.0005      -0.0002       0.0018
COOLANTTEMP           0.3907       0.0245       0.3428       0.4387
HUMIDITY              0.0807       0.0094       0.0623       0.0991
FEEDRATE              5.9122       3.6069      -1.1572      12.9817


     Faktor  Sklon @ 


NOTE: ODS OUTPUT: PARAMETERESTIMATES -> pe
NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: DATA work.compare

NOTE: Stream 1 processed 6 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 6 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote work.compare (6 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=work.compare

NOTE: PROC PRINT completed: 6 observations printed, 4 variables


## Krok 4 — Ohodnocení každého dílu na 90. percentilu

Příkaz `OUTPUT` zapisuje predikci 90. percentilu zpět, jeden řádek na díl, spolu se směrodatnou chybou predikce (`STDP`) a reziduálem kontrolní ztráty. `PartID` provázíme příkazem `ID` a kopírujeme dva dominantní faktory, aby analytici mohli seřadit nejrizikovější díly nahoru. Malý PROC PRINT ukazuje první ohodnocené díly.

In [5]:
PROCEDURA quantreg data=work.machining ci=sparsity;
    TŘÍDA Machine Shift;
    id PartID;
    MODEL Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      FeedRate
        / quantile=0.90;
    VÝSTUP out=work.scored
        predicted=PredP90
        stdp=SEPred
        residual=Resid;
SPUSTIT;

PROCEDURA TISK data=work.scored(obs=10) noobs ŠTÍTEK;
    PROMĚNNÁ PartID Machine ToolWear CycleSpeed PredP90 SEPred Resid;
    ŠTÍTEK PartID     = 'ID dílu'
          Machine    = 'Stroj'
          ToolWear   = 'Opotřebení nástroje (min)'
          CycleSpeed = 'Rychlost cyklu (ot/min)'
          PredP90 = 'Predikovaná 90. percentilová odchylka'
          SEPred  = 'Směrodatná chyba predikce'
          Resid   = 'Reziduum kontrolní ztráty';
SPUSTIT;


The QUANTREG Procedure

Quantile: 0.9000
CI Method: SPARSITY
Dependent Variable: Deviation

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -3.9687       0.6322      -5.2078      -2.7295
MACHINE M3            0.6729       0.1246       0.4287       0.9171
MACHINE M2           -0.4499       0.1117      -0.6689      -0.2310
MACHINE M1           -1.1957       0.1109      -1.4131      -0.9784
SHIFT EVE            -3.1741       0.1034      -3.3768      -2.9713
SHIFT DAY            -1.8083       0.1017      -2.0075      -1.6090
TOOLWEAR              0.0438       0.0012       0.0416       0.0461
CYCLESPEED            0.0037       0.0003       0.0032       0.0043
COOLANTTEMP           0.3377       0.0133       0.3116       0.3638
FEEDRATE             14.9464       2.0482      10.9321      18.9608


 ID dílu     Opotřebení nástroje (min)  Rychlost cyklu (ot/min)    Predikovaná 90. percentilová odchylka    Směrodatná chyba predikce    Reziduum kontrolní z


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: PROC PRINT data=work.scored

NOTE: PROC PRINT completed: 10 observations printed, 6 variables


## Interpretace výsledků

**Chvost vypráví jiný příběh než střed.** Porovnáním obou bloků koeficientů (Krok 2) a sloučené tabulky (Krok 3) jsou sklony pro `ToolWear`, `CycleSpeed` a `FeedRate` výrazně vyšší na 90. percentilu než na mediánu. To je viditelný projev generujícího mechanismu: protože jsme sestrojili rozptyl chyby tak, aby rostl s opotřebením a rychlostí, tyto faktory sotva posouvají medián odchylky, ale silně nafukují horní chvost náchylný ke zmetkovitosti. Regrese založená na průměru by právě tyto páky podhodnotila.

**Signál `ToolWear` je nejjasnější.** Jeho sklon se zhruba ztrojnásobí z hodnoty pro medián (0,015) na hodnotu pro chvost (0,042) a interval spolehlivosti pro 90. percentil leží daleko nad nulou — opotřebení je nejspolehlivějším jednotlivým faktorem vysokého rizika chvostu. `CycleSpeed`, na mediánu prakticky plochý, se v chvostu stává kladným, což odpovídá jeho roli v členu rozptylu.

**Kvantilová regrese odděluje polohu od rozptylu.** `CoolantTemp` vstupuje do členu polohy, ale ne do členu rozptylu, takže její sklon zůstává blízko 0,4–0,5 mikronu na stupeň u obou kvantilů — posouvá celé rozdělení, místo aby rozevírala chvost. `Humidity` nenese v generujícím procesu žádný skutečný signál, přesto na pouhých 100 dílech získává malý zdánlivý sklon; její změna `Tail - Median` (0,013) je o řád menší než u `ToolWear` (0,027) a mnohonásobně menší než u `FeedRate` (12,3), takže se nechová jako faktor chvostu. Poctivé poučení je, že proměnná bez skutečného efektu může na malém vzorku stále vypadat nenulově — právě proto by licencovaný produkční běh přes tisíce dílů tyto odhady zpřesnil.

**Provozní přínos.** Predikce z `OUTPUT` dávají každému dílu predikovaný 90. percentil odchylky se směrodatnou chybou a označují díly s vysokým rizikem chvostu ještě před expedicí. Praktický závěr: zkrátit intervaly výměny nástroje a omezit rychlost cyklu při zakázkách s přísnými tolerancemi, protože obě opatření působí přímo na horní chvost rozměrové odchylky způsobující zmetky.

*Poznámka k rozsahu:* tento sešit běží pod nelicencovaným enginem, který omezuje každý krok na 100 pozorování, takže výše uvedené sklony jsou odhadnuty na 100 dílech a proložení chvostu je nutně šumnější, než by bylo u plně produkčního běhu. Vzorec střed-versus-chvost je vidět již při této velikosti; licencovaný běh přes celý proud dílů by zpřesnil každý interval spolehlivosti.